# 01 —  Radar Dataset Exploration and Structural Validation

This notebook explores the SAAB SIRS 1600 radar dataset containing measurements of drones, birds, humans, and a corner reflector.

The objectives are to:

1. Validate the structure of the dataset.
2. Inspect the radar measurements and associated metadata.
3. Analyse the distribution of recording sessions and radar segments.
4. Identify class imbalance and potential data-splitting limitations.
5. Prepare the dataset for drone-versus-bird classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
FILE_PATH = Path("../data/raw/data_SAAB_SIRS_77GHz_FMCW.npy")

print("Current working directory:", Path.cwd())
print("Resolved dataset path:", FILE_PATH.resolve())
print("Dataset file found:", FILE_PATH.exists())

## 1. Dataset Loading

The dataset is stored as a NumPy object array because each measurement session contains a different number of radar segments. The option `allow_pickle=True` is required to load this structure.

In [ ]:
if not FILE_PATH.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {FILE_PATH.resolve()}"
    )

data = np.load(FILE_PATH, allow_pickle=True)

print("Dataset shape:", data.shape)
print("Dataset dtype:", data.dtype)

## 2. Initial Dataset Structure Inspection

Each of the 130 rows represents one radar measurement session. The six columns contain the target label, complex radar segments, range values, timestamps, official data-split indicators, and field-of-view edge indicators.

In [ ]:
def extract_label(value):
    """Extract a clean string label from a dataset label field."""
    label_array = np.asarray(value).reshape(-1)

    if label_array.size == 0:
        return "unknown"

    return str(label_array[0]).strip()

In [ ]:
row = data[0]

label = extract_label(row[0])
segments = np.asarray(row[1])

ranges = np.asarray(row[2]).reshape(-1)
times = np.asarray(row[3]).reshape(-1)
splits = np.asarray(row[4]).reshape(-1)
edge_flags = np.asarray(row[5]).reshape(-1)

print("Target label:", label)
print("Radar segments shape:", segments.shape)
print("Range values shape:", ranges.shape)
print("Time values shape:", times.shape)
print("Data split indicators shape:", splits.shape)
print("Field-of-view edge indicators shape:", edge_flags.shape)

print("\nFirst five range values (m):", ranges[:5])
print("First five timestamps (s):", times[:5])
print("First five data split indicators:", splits[:5])
print("First five edge indicators:", edge_flags[:5])

print("\nMetadata consistency check:")
print("Radar segments:", segments.shape[1])
print("Range values:", len(ranges))
print("Timestamps:", len(times))
print("Split indicators:", len(splits))
print("Edge indicators:", len(edge_flags))

### First-Session Inspection Results

The first dataset row corresponds to target **D1**, representing the **DJI Matrice 200 V drone**.

The radar data matrix has a shape of **(1280, 1228)**:

- The 1,280 rows represent five range cells containing 256 complex-valued measurements each.
- The 1,228 columns represent the radar scan segments collected during this measurement session.
- Each segment can be reshaped into a complex-valued matrix of shape **(5, 256)**.

Each metadata array also contains 1,228 elements:

| Metadata field | Number of elements |
|---|---:|
| Radar segments | 1,228 |
| Range values | 1,228 |
| Timestamps | 1,228 |
| Data split indicators | 1,228 |
| Field-of-view edge indicators | 1,228 |

The first five segments were recorded at a target-centre range of approximately **62.26 m**. The timestamps show an interval of approximately **0.1 s** between successive segments.

The split indicators are defined as:

- `1`: Training set
- `2`: Validation set
- `3`: Test set

The first five split indicators are `[2, 3, 1, 3, 1]`. This indicates that segments from the same measurement session can be assigned to different data partitions.

The first five edge indicators are all zero, meaning that these target observations were not truncated at the boundary of the radar's field of view.

### Conclusion

The first measurement session is structurally consistent. The radar matrix contains 1,228 segments, and every associated metadata array contains exactly 1,228 values.

## 3. Analysis of All Measurement Sessions

The following analysis validates the structure of all 130 measurement sessions and creates a session-level metadata table.

In [ ]:
session_records = []
structural_problems = []

for session_id, row in enumerate(data):
    label = extract_label(row[0])
    segments = np.asarray(row[1])

    ranges = np.asarray(row[2]).reshape(-1)
    times = np.asarray(row[3]).reshape(-1)
    splits = np.asarray(row[4]).reshape(-1)
    edge_flags = np.asarray(row[5]).reshape(-1)

    if segments.ndim != 2:
        structural_problems.append(
            f"Session {session_id}: segments.ndim={segments.ndim}"
        )
        continue

    n_segments = segments.shape[1]

    metadata_lengths = {
        "ranges": len(ranges),
        "times": len(times),
        "splits": len(splits),
        "edge_flags": len(edge_flags)
    }

    if segments.shape[0] != 1280:
        structural_problems.append(
            f"Session {session_id}: segments.shape={segments.shape}"
        )

    for field_name, field_length in metadata_lengths.items():
        if field_length != n_segments:
            structural_problems.append(
                f"Session {session_id}: "
                f"{field_name}={field_length}, "
                f"segments={n_segments}"
            )

    session_records.append({
        "session_id": session_id,
        "label": label,
        "n_segments": n_segments,
        "min_range_m": float(np.min(ranges)),
        "max_range_m": float(np.max(ranges)),
        "duration_s": float(np.max(times) - np.min(times)),
        "train_samples": int(np.sum(splits == 1)),
        "validation_samples": int(np.sum(splits == 2)),
        "test_samples": int(np.sum(splits == 3)),
        "edge_samples": int(np.sum(edge_flags == 1))
    })

sessions_df = pd.DataFrame(session_records)

print("Number of measurement sessions:", len(sessions_df))
print("Total number of radar segments:", sessions_df["n_segments"].sum())
print("Number of structural problems:", len(structural_problems))

if structural_problems:
    print("\nFirst detected structural problems:")
    print(*structural_problems[:10], sep="\n")

display(sessions_df.head())

### Structural Validation Results

The dataset contains **130 measurement sessions** and **75,868 radar segments**.

No structural problems were detected. Every session contains a two-dimensional radar matrix with 1,280 rows, and every metadata array has the same number of elements as the corresponding number of radar segments.

### Conclusion

The complete dataset is structurally valid and can proceed to class-distribution analysis and signal preprocessing.

## 4. Class Distribution Analysis

The following table presents the number of sessions and radar segments associated with each original target label.

In [ ]:
class_summary = (
    sessions_df
    .groupby("label")
    .agg(
        sessions=("session_id", "nunique"),
        segments=("n_segments", "sum"),
        train=("train_samples", "sum"),
        validation=("validation_samples", "sum"),
        test=("test_samples", "sum"),
        edge_samples=("edge_samples", "sum")
    )
    .sort_values("segments", ascending=False)
)

display(class_summary)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

REPORT_FIGURE_DIR = (
    Path("../outputs/report_assets/figures")
)

REPORT_TABLE_DIR = (
    Path("../outputs/report_assets/tables")
)

REPORT_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DRONE_LABELS = {
    "D1", "D2", "D3",
    "D4", "D5", "D6"
}

BIRD_LABELS = {
    "black-headed gull",
    "seagull",
    "seagull and black-headed gull",
    "heron",
    "pigeon",
    "raven"
}

subtype_composition_df = (
    class_summary
    .reset_index()
    .rename(columns={
        "label": "original_label",
        "segments": "total_segments"
    })
)

subtype_composition_df[
    "target_group"
] = subtype_composition_df[
    "original_label"
].map(
    lambda label: (
        "Drone"
        if label in DRONE_LABELS
        else (
            "Bird"
            if label in BIRD_LABELS
            else "Excluded"
        )
    )
)

subtype_composition_df[
    "usable_segments"
] = (
    subtype_composition_df[
        "total_segments"
    ]
    - subtype_composition_df[
        "edge_samples"
    ]
)

binary_subtype_df = (
    subtype_composition_df[
        subtype_composition_df[
            "target_group"
        ].isin([
            "Bird",
            "Drone"
        ])
    ]
    .sort_values(
        "usable_segments",
        ascending=True
    )
)

binary_subtype_df.to_csv(
    REPORT_TABLE_DIR
    / "dataset_subtype_composition.csv",
    index=False
)

colour_map = {
    "Bird": "#4C9F70",
    "Drone": "#3B6FB6"
}

fig, axis = plt.subplots(
    figsize=(10, 7)
)

axis.barh(
    binary_subtype_df[
        "original_label"
    ],
    binary_subtype_df[
        "usable_segments"
    ],
    color=[
        colour_map[group]
        for group
        in binary_subtype_df[
            "target_group"
        ]
    ]
)

axis.set_xlabel(
    "Usable radar segments"
)

axis.set_ylabel(
    "Original target subtype"
)

axis.set_title(
    "Usable Bird and Drone Segments by Target Subtype"
)

axis.grid(
    axis="x",
    alpha=0.2
)

fig.tight_layout()

figure_path = (
    REPORT_FIGURE_DIR
    / "dataset_subtype_distribution.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Figure saved:", figure_path.resolve())

### Class Distribution Results

The six drone classes contain a combined total of **58,768 segments from 44 measurement sessions**. The bird categories contain **7,792 segments from 56 sessions**.

The dataset therefore contains approximately **7.5 times more drone segments than bird segments**, indicating a substantial class imbalance.

The bird categories are also internally imbalanced. The dataset contains 4,364 black-headed gull segments and 2,550 seagull segments, but only 32 pigeon segments and 19 raven segments. These rare categories cannot support reliable species-level classification.

For the initial binary classification experiment:

- `D1` to `D6` will be merged into the `drone` class.
- All bird categories will be merged into the `bird` class.
- Human and corner-reflector observations will be excluded.
- The 67 field-of-view edge samples will initially be excluded.

The six drone types contain between 6,921 and 12,093 segments each. Therefore, fine-grained drone-model classification can later be investigated as a secondary experiment.

### Evaluation Implications

Accuracy alone could be misleading because of the substantial drone–bird imbalance. The primary evaluation metrics will therefore include:

- Macro-F1 score
- Balanced accuracy
- Precision and recall for each class
- Confusion matrix

The official partition distributes segments from the same measurement session across training, validation, and testing. This may introduce dependency between the partitions because temporally adjacent segments can be highly similar.

The project will initially use the official split for comparability with the original study. A stricter session-independent evaluation will subsequently be considered to measure generalisation to unseen recording sessions.